# NanowireML — Full Pipeline

Reproducing and extending Raya et al. 2025: sequence-only ML prediction of microbial nanowire proteins.

**Run all cells** — the notebook handles setup, feature extraction, baseline reproduction,
hyperparameter tuning, calibration, SHAP, feature ablation, and the autoresearch loop.

Features are persisted to Google Drive so reconnecting doesn't require re-extraction.

> Tip: Use a **T4 GPU runtime** (Runtime → Change runtime type → T4 GPU) for faster tuning.
> The pipeline is CPU-compatible but XGBoost tuning is 5-10x faster on GPU.

## Setup

In [ ]:
# Clone repo and install dependencies
!git clone https://github.com/moneet-dev/nanowire-ml.git 2>/dev/null || (cd nanowire-ml && git pull)
%cd nanowire-ml

!pip install -q -r requirements.txt 2>/dev/null
!pip install -q --no-deps iFeatureOmegaCLI
!pip install -q networkx rdkit 2>/dev/null

# ESM-2 dependencies (GPU required)
!pip install -q fair-esm torch 2>/dev/null

# Verify GPU
import torch
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    print(f'\nGPU: {gpu} ({torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB)')
else:
    print('\nWARNING: No GPU detected. Switch to T4 GPU: Runtime → Change runtime type → T4 GPU')

print('=== Setup complete ===')

In [ ]:
# Mount Google Drive and symlink features directory for persistence
from google.colab import drive
import os

drive.mount('/content/drive')
feat_dir = '/content/drive/MyDrive/nanowire-ml/features'
os.makedirs(feat_dir, exist_ok=True)

local_feat = 'data/features'
if os.path.islink(local_feat):
    os.unlink(local_feat)
elif os.path.isdir(local_feat):
    import shutil
    shutil.rmtree(local_feat)
os.symlink(feat_dir, local_feat)

print(f'Features dir -> {feat_dir}')

## Phase 1 — Data Preparation

In [ ]:
!python scripts/01_data_prep.py

## Phase 2 — Feature Extraction

Extracts 22 iFeature descriptor groups. First run takes ~3 min; subsequent runs skip if `master_features.csv` already exists on Drive.

In [ ]:
import os

nr_exists = os.path.exists('data/features/master_features.csv')
red_exists = os.path.exists('data/features/master_features_redundant.csv')

if nr_exists and red_exists:
    print('Both feature matrices already exist on Drive — skipping extraction.')
else:
    variants = []
    if not nr_exists:
        variants.append('nr')
    if not red_exists:
        variants.append('redundant')
    cmd = 'python scripts/02_feature_extraction.py ' + ' '.join(variants)
    print(f'Running: {cmd}')
    !{cmd}

## Phase 3 — Reproduce Baseline (Raya et al. 2025 Table 2)

In [ ]:
# Reproduce on redundant (paper-faithful) set
!python scripts/03_reproduce_baseline.py redundant

In [ ]:
# Reproduce on non-redundant set
!python scripts/03_reproduce_baseline.py nr

In [ ]:
# Display reproduction comparison
import pandas as pd

print('=== Redundant (paper-faithful) ===')
display(pd.read_csv('results/metrics/reproduction_table.csv'))

print('\n=== Non-redundant ===')
display(pd.read_csv('results/metrics/reproduction_table_nr.csv'))

## Phase 4 — Extended Pipeline

Novel contributions: hyperparameter tuning (RandomizedSearchCV, 50 iter), calibration (Platt + isotonic), stacking ensemble, SHAP attribution, bootstrap CIs.

**This is the longest cell** — ~15 min on Colab CPU, ~5 min on T4 GPU.

In [ ]:
!python -u scripts/04_extended_pipeline.py nr

In [ ]:
# Display extended results
import pandas as pd
print('=== Extended Pipeline Results ===')
display(pd.read_csv('results/metrics/extended_table.csv'))

print('\n=== Best Hyperparameters ===')
import json
with open('results/metrics/best_params.json') as f:
    print(json.dumps(json.load(f), indent=2))

In [ ]:
# Display figures
from IPython.display import Image, display
import os

fig_dir = 'results/figures'
for fig in ['roc_curves.png', 'pr_curves.png', 'calibration_plots.png', 'shap_summary.png']:
    path = os.path.join(fig_dir, fig)
    if os.path.exists(path):
        print(f'\n--- {fig} ---')
        display(Image(filename=path, width=700))

## Phase 5 — ESM-2 Protein Language Model

Generate 1280-dimensional ESM-2 embeddings (Meta's protein language model) for each sequence, then run the same classifier battery. This produces the **iFeature vs ESM-2 comparison table** — a core research finding.

**Requires T4 GPU.** Embedding ~1664 sequences takes ~5 min on T4; classifiers are fast on 1280 features.

In [ ]:
!python -u scripts/04b_esm2_pipeline.py

In [ ]:
# Display ESM-2 results and comparison
import pandas as pd
from IPython.display import Image, display
import os

print('=== ESM-2 Results ===')
if os.path.exists('results/metrics/esm2_table.csv'):
    display(pd.read_csv('results/metrics/esm2_table.csv'))

print('\n=== ESM-2 vs iFeature Comparison ===')
if os.path.exists('results/metrics/esm2_vs_ifeature.csv'):
    display(pd.read_csv('results/metrics/esm2_vs_ifeature.csv'))

if os.path.exists('results/figures/esm2_roc_pr_curves.png'):
    print('\n--- ESM-2 ROC + PR curves ---')
    display(Image(filename='results/figures/esm2_roc_pr_curves.png', width=700))

## Phase 6 — Feature Ablation Study

Tests each descriptor group individually and in biologically motivated combinations.

In [ ]:
!python -u scripts/05_ablation.py nr

In [ ]:
import pandas as pd
from IPython.display import Image, display

print('=== Ablation Results ===')
df = pd.read_csv('results/metrics/ablation_table.csv')
display(df.sort_values('roc_auc', ascending=False))

if os.path.exists('results/figures/ablation_bar.png'):
    display(Image(filename='results/figures/ablation_bar.png', width=700))

## Phase 7 — Autoresearch Loop

Runs 10 predefined experiments using the keep/revert pattern from Karpathy's autoresearch framework.

In [ ]:
!python -u scripts/06_autoresearch_loop.py

In [ ]:
import pandas as pd

print('=== Autoresearch Experiment Log ===')
display(pd.read_csv('results/metrics/autoresearch_log.csv'))

## Push Results to GitHub

Commit all result CSVs and push to the repo. You'll need a GitHub Personal Access Token (PAT) with `repo` scope.

**Create one at:** Settings → Developer settings → Personal access tokens → Generate new token (classic) → check `repo` scope.

In [ ]:
from getpass import getpass
token = getpass('GitHub Personal Access Token (repo scope): ')

!git config user.email "devadigamoneet@gmail.com"
!git config user.name "moneet-dev"

import subprocess
subprocess.run(['git', 'remote', 'set-url', 'origin',
                f'https://{token}@github.com/moneet-dev/nanowire-ml.git'],
               capture_output=True)

!git add results/metrics/ scripts/ src/ notebooks/
!git status

!git commit -m "results: Phases 3-7 from Colab — reproduction, tuning, calibration, SHAP, ablation, autoresearch"
!git push origin main

# Clear token from remote URL
!git remote set-url origin https://github.com/moneet-dev/nanowire-ml.git
print('\n=== Pushed to GitHub ===')

## Summary

All phases complete. Check the results:

In [ ]:
import os
print('=== Result files ===')
for f in sorted(os.listdir('results/metrics')):
    if f.endswith('.csv') or f.endswith('.json'):
        size = os.path.getsize(f'results/metrics/{f}')
        print(f'  {f:45s} {size:>8,d} bytes')

print('\n=== Figures ===')
fig_dir = 'results/figures'
if os.path.exists(fig_dir):
    for f in sorted(os.listdir(fig_dir)):
        if f.endswith('.png'):
            size = os.path.getsize(f'{fig_dir}/{f}')
            print(f'  {f:45s} {size:>8,d} bytes')